In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("pdfs/linkedin.pdf")
linkedin = ''
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
with open("pdfs/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [7]:
# tools provide extended capabilities to agents
# below is the tools schema, every tool must be json defined like this
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }

}

In [8]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [9]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email Received"


In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model='gpt-5-nano', messages=messages, tools=tools)
    
    while response.choices[0].finish_reason=='tool_calls':
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
           email = json.loads(tool_call.function.arguments).get('email')
           record_email_tool(email)
           messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model='gpt-5-nano', messages=messages, tools=tools)

    return response.choices[0].message.content


In [17]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: x@x.com
Tool called to record an email: y@y.com
Tool called to record an email: z@z.com
